# 📦 Telegram Smart Compressor
ارمي الملفات في قناة **📦 Smart Compressor** واضغط ▶. أول تشغيل فقط سيطلب بيانات Telegram ويصنع القناة تلقائيًا.

**إعادة معالجة نتيجة قديمة:** اعمل Reply على الملف بـ `🔁`، أو `🔁 أصغر`، `🔁 صوت`، `🔁 80`.


In [ ]:
#@title 🚀 Smart Compressor — اضغط ▶ فقط
#@markdown في الاستخدام العادي: سيب PROFILE = SMART_AUTO واضغط ▶.
#@markdown TARGET_SIZE_MB اختياري ويُستخدم فقط مع VIDEO_TARGET_SIZE.
PROFILE = "SMART_AUTO" #@param ["SMART_AUTO", "AUDIO_TINY", "VIDEO_BALANCED", "VIDEO_FAST", "VIDEO_SMALLEST", "VIDEO_TARGET_SIZE"]
TARGET_SIZE_MB = "" #@param {type:"string"}

import os, re, subprocess, urllib.request, json, time
REPO = "abdullahsamirashour/gpt"
BRANCH = "main"
ENGINE_REL = "telegram-smart-compressor/engine.py"
ENGINE_PATH = "/content/telegram_smart_compressor_engine.py"
os.environ["TSC_PROFILE"] = str(PROFILE or "SMART_AUTO")
os.environ["TSC_TARGET_SIZE_MB"] = str(TARGET_SIZE_MB or "").strip()
os.environ["TSC_UPDATE_CHANNEL"] = BRANCH

def latest_sha():
    p = subprocess.run(["git","ls-remote",f"https://github.com/{REPO}.git",f"refs/heads/{BRANCH}"],capture_output=True,text=True,timeout=30)
    if p.returncode == 0 and p.stdout.strip():
        sha=p.stdout.strip().split()[0]
        if re.fullmatch(r"[0-9a-f]{40}",sha): return sha
    req=urllib.request.Request(f"https://api.github.com/repos/{REPO}/git/ref/heads/{BRANCH}?t={int(time.time())}",headers={"Accept":"application/vnd.github+json","User-Agent":"Smart-Compressor-Colab","Cache-Control":"no-cache"})
    with urllib.request.urlopen(req,timeout=30) as r: return json.loads(r.read().decode())["object"]["sha"]

def download(sha):
    url=f"https://raw.githubusercontent.com/{REPO}/{sha}/{ENGINE_REL}"
    req=urllib.request.Request(url,headers={"User-Agent":"Smart-Compressor-Colab","Cache-Control":"no-cache"})
    with urllib.request.urlopen(req,timeout=30) as r: return r.read().decode("utf-8")

print("🌐 Checking latest stable engine...")
sha=latest_sha(); print("🔗 Commit:",sha[:10])
engine=download(sha)
m=re.search(r'ENGINE_BUNDLE_VERSION\s*=\s*"([^"]+)"',engine)
print("📦 Engine:",m.group(1) if m else "unknown")
if len(engine)<5000 or "ENGINE_BUNDLE_VERSION" not in engine: raise RuntimeError("Engine download invalid")
open(ENGINE_PATH,"w",encoding="utf-8").write(engine)
print("▶️ Starting...")
print()
exec(compile(engine,ENGINE_PATH,"exec"),globals(),globals())
